# 01 Build the dataset

**Goal:** one tidy table with **one row per NBA draft pick from 1989 to 2018**.
Each row has what was known on draft night, plus what the player went on to do.
Every later notebook reads the file this notebook saves: `data/processed/draft_players.csv`.

The raw files in `data/` are only read, never changed.

| Step | What happens |
|---|---|
| 1 | Load the draft picks (one row per pick) |
| 2 | Add facts about the player: age at draft, height, weight, position |
| 3 | Clean the season-by-season stats (fix traded players being counted twice) |
| 4 | Calculate career outcomes for each pick, giving 0 to picks who never played |
| 5 | Check the table automatically and save it |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# never change these settings!
RAW = Path("../data/archive")
PROCESSED = Path("../data/processed")

FIRST_DRAFT = 1989
LAST_DRAFT = 2018    # every pick has had at least 4 full seasons since
FIRST_N_SEASONS = 4  # roughly the length of a rookie contract

## Load the draft picks

Source: `Draft Pick History.csv`, which has one row per pick.

- I keep **NBA** drafts from 1989 to 2018. The file also includes the old BAA league.
- In this file, `season` means the year of the draft. For example, `2010` is the June 2010 draft.
- Each pick has a `player_id`, a code that is unique to each player. I join files on this ID
  instead of on the player's name, because different players can share a name.
- A missing `college` usually means an international or high-school player, so I keep that
  as a category (`no_college`) instead of treating it as an error.

In [16]:
draft_raw = pd.read_csv(RAW / "Draft Pick History.csv")

draft = draft_raw[draft_raw["season"].between(FIRST_DRAFT, LAST_DRAFT)].copy()
draft = draft.rename(columns={"season": "draft_year", "tm": "draft_team", "lg": "NBA"})
draft = draft[["player_id", "player", "draft_year", "overall_pick", "round", "draft_team", "college"]]

draft[["overall_pick", "round"]] = draft[["overall_pick", "round"]].astype(int)
draft["background"] = np.where(draft["college"].notna(), "college", "no_college")

print("Picks:", len(draft))
print("Picks per draft: min", draft.groupby("draft_year").size().min(), "/ max", draft.groupby("draft_year").size().max())
print(draft["background"].value_counts().to_string())
draft.head()

Picks: 1742
Picks per draft: min 54 / max 60
background
college       1433
no_college     309


,player_id,player,draft_year,overall_pick,round,draft_team,college,background
473,aytonde01,Deandre Ayton,2018,1,1,PHO,Arizona,college
474,baglema01,Marvin Bagley III,2018,2,1,SAC,Duke,college
475,doncilu01,Luka Dončić,2018,3,1,ATL,NaN,no_college
476,jacksja02,Jaren Jackson Jr.,2018,4,1,MEM,Michigan State,college
477,youngtr01,Trae Young,2018,5,1,DAL,Oklahoma,college


## Add facts about the player

Source: `Player Career Info.csv`, which has one row per player.

- **Age at draft** = draft day minus birth date. I treat **June 25** as draft day every year,
  since the draft is always in late June, so the age is off by a few days at most.
- **Height, weight and position** are the player's listed career values, not measurements taken
  on draft night. They are close, but not exact.
- `pos_main` keeps only the first position listed, so `F-C` becomes `F`.

**Important:** this file only lists players who played at least one NBA game. The picks who never
played get **blank** values here. They are **kept** in the table, not removed (step 4 explains why).

In [18]:
info = pd.read_csv(RAW / "Player Career Info.csv")
info = info.rename(columns={"ht_in_in": "height_in", "wt": "weight_lb"})
info = info[["player_id", "birth_date", "height_in", "weight_lb", "pos"]]

draft = draft.merge(info, on="player_id", how="left")

draft_day = pd.to_datetime(draft["draft_year"].astype(str) + "-06-25")
birth = pd.to_datetime(draft["birth_date"])
draft["age_at_draft"] = ((draft_day - birth).dt.days / 365.25).round(1)
draft["pos_main"] = draft["pos"].str.split("-").str[0]

print("Picks with no career info (blank age/height/position):", draft["birth_date"].isna().sum())
draft[["player", "draft_year", "overall_pick", "age_at_draft", "height_in", "weight_lb", "pos_main"]].head()

Picks with no career info (blank age/height/position): 240


,player,draft_year,overall_pick,age_at_draft,height_in,weight_lb,pos_main
0,Deandre Ayton,2018,1,19.9,84.0,252.0,C
1,Marvin Bagley III,2018,2,19.3,82.0,235.0,F
2,Luka Dončić,2018,3,19.3,80.0,230.0,G
3,Jaren Jackson Jr.,2018,4,18.8,82.0,242.0,F
4,Trae Young,2018,5,19.8,74.0,164.0,G


## Clean the season-by-season stats

Source: `Advanced.csv`, which has one row per player, per season, per team.

- I keep **NBA** seasons only. The file also contains the old ABA and BAA leagues.
- **Traded players:** if a player was traded mid-season, the file has a **season-total row**
  (team `2TM`, `3TM`, ...) **and** one row for each team. Adding up every row counts that season
  twice. This was a bug in the original project. Fix: when a total row exists, keep only that row.
  The result is **exactly one row per player per season**.
- In this file, `season` means the year the season **ends**. A player drafted in June 2010 plays
  their first season in `2011`.

In [21]:
adv = pd.read_csv(RAW / "Advanced.csv", usecols=["season", "lg", "player_id", "team", "g", "mp", "ws"])
adv = adv[adv["lg"] == "NBA"].copy()
adv["is_total_row"] = adv["team"].str.fullmatch(r"\d+TM")

# Every multi-team season should have a total row; check before relying on it
rows_per_season = adv.groupby(["player_id", "season"])["is_total_row"].agg(["size", "any"])
assert not ((rows_per_season["size"] > 1) & ~rows_per_season["any"]).any(), "multi-team season without a total row"

# Total rows sort first, so drop_duplicates keeps them
seasons = (
    adv.sort_values("is_total_row", ascending=False)
    .drop_duplicates(["player_id", "season"])
    .drop(columns=["lg", "is_total_row"])
)

print("Rows before:", len(adv), "after:", len(seasons))

Rows before: 31560 after: 25760


Example: Žan Tabak (picked in 1991) was traded during the 1998 season.
Before the fix, 1998 appears three times. After the fix, it appears once (the `2TM` row).

In [5]:
tabak = draft.loc[draft["player"] == "Žan Tabak", "player_id"].iloc[0]
print("Before:")
print(adv[adv["player_id"] == tabak].sort_values("season")[["season", "team", "g", "ws"]].to_string(index=False))
print("\nAfter:")
print(seasons[seasons["player_id"] == tabak].sort_values("season")[["season", "team", "g", "ws"]].to_string(index=False))

Before:
 season team  g   ws
   1995  HOU 37  0.2
   1996  TOR 67  1.1
   1997  TOR 13  0.1
   1998  2TM 57  0.0
   1998  TOR 39 -0.2
   1998  BOS 18  0.2
   2000  IND 18  0.1
   2001  IND 55  0.9

After:
 season team  g  ws
   1995  HOU 37 0.2
   1996  TOR 67 1.1
   1997  TOR 13 0.1
   1998  2TM 57 0.0
   2000  IND 18 0.1
   2001  IND 55 0.9


## Calculate outcomes for each pick

For each drafted player:

| Column | Meaning |
|---|---|
| `ws_first4` | **Main target.** Win shares in the 4 seasons right after the draft (`draft_year + 1` to `draft_year + 4`), whether or not the player actually played in them. This is roughly the rookie contract, the years the drafting team controls. |
| `games_first4` | Games played in those same 4 seasons |
| `ws_career`, `games_career`, `seasons_played` | Career totals in the data, kept for context |
| `played_nba` | True if the player appeared in at least one NBA game |

**Picks who never played get 0, not removed.** If I dropped them, late picks would look better
than they really were, because their failures would disappear from the data.
This is called **survivorship bias**, and the original project had it.

Note: career totals for players who are still active only run to the end of the data (it includes
the current season), so `ws_career` keeps growing for them. `ws_first4` is complete for everyone,
because the last window ends in 2022.

In [6]:
s = seasons.merge(draft[["player_id", "draft_year"]], on="player_id", how="inner")
s["years_after_draft"] = s["season"] - s["draft_year"]

first_n = (
    s[s["years_after_draft"].between(1, FIRST_N_SEASONS)]
    .groupby("player_id")
    .agg(ws_first4=("ws", "sum"), games_first4=("g", "sum"))
)
career = s.groupby("player_id").agg(
    ws_career=("ws", "sum"), games_career=("g", "sum"), seasons_played=("season", "nunique")
)

draft = draft.merge(first_n, on="player_id", how="left").merge(career, on="player_id", how="left")
draft["played_nba"] = draft["games_career"].notna()

outcome_cols = ["ws_first4", "games_first4", "ws_career", "games_career", "seasons_played"]
draft[outcome_cols] = draft[outcome_cols].fillna(0)
draft[["games_first4", "games_career", "seasons_played"]] = draft[["games_first4", "games_career", "seasons_played"]].astype(int)

print("Played in the NBA:", draft["played_nba"].sum(), "| never played:", (~draft["played_nba"]).sum())
draft[["player", "draft_year", "overall_pick", "ws_first4", "ws_career", "seasons_played", "played_nba"]].head()

Played in the NBA: 1502 | never played: 240


,player,draft_year,overall_pick,ws_first4,ws_career,seasons_played,played_nba
0,Deandre Ayton,2018,1,24.6,44.0,9,True
1,Marvin Bagley III,2018,2,7.4,16.7,9,True
2,Luka Dončić,2018,3,29.0,67.5,9,True
3,Jaren Jackson Jr.,2018,4,12.9,30.4,9,True
4,Trae Young,2018,5,26.4,45.5,9,True


## Check the table automatically and save it

The notebook stops with an error if anything is wrong:

- one row per pick, with no player appearing twice
- every pick number is between 1 and 60
- age, height and position are blank **only** for picks who never played (the issue found in step 2)
- picks who never played have exactly 0 games and 0 win shares

In [7]:
expected_rows = len(draft_raw[(draft_raw["lg"] == "NBA") & draft_raw["season"].between(FIRST_DRAFT, LAST_DRAFT)])
assert len(draft) == expected_rows, "row count changed during merges"
assert draft["player_id"].is_unique, "a player appears twice"
assert draft["overall_pick"].between(1, 60).all(), "pick number out of range"
assert (draft["birth_date"].isna() == ~draft["played_nba"]).all(), "blank info does not match never-played picks"
assert (draft.loc[~draft["played_nba"], ["ws_first4", "ws_career", "games_career"]] == 0).all().all()

columns = [
    "player_id", "player", "draft_year", "overall_pick", "round", "draft_team", "college", "background",
    "age_at_draft", "height_in", "weight_lb", "pos_main", "played_nba",
    "ws_first4", "games_first4", "ws_career", "games_career", "seasons_played",
]
final = draft[columns].sort_values(["draft_year", "overall_pick"]).reset_index(drop=True)

PROCESSED.mkdir(parents=True, exist_ok=True)
final.to_csv(PROCESSED / "draft_players.csv", index=False)
print("All checks passed. Saved", final.shape, "to", PROCESSED / "draft_players.csv")

All checks passed. Saved (1742, 18) to ..\data\processed\draft_players.csv


In [8]:
final.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
draft_year,1742.0,2003.80,8.60,1989.0,1996.0,2004.0,2011.00,2018.0
overall_pick,1742.0,29.63,16.88,1.0,15.0,30.0,44.00,60.0
round,1742.0,1.50,0.50,1.0,1.0,2.0,2.00,2.0
age_at_draft,1502.0,21.57,1.40,17.7,20.6,21.8,22.50,27.4
height_in,1502.0,79.13,3.58,67.0,77.0,79.5,82.00,91.0
weight_lb,1502.0,221.24,27.49,140.0,200.0,220.0,240.00,325.0
ws_first4,1742.0,6.03,8.25,-1.8,0.0,2.3,9.57,55.3
games_first4,1742.0,145.79,111.47,0.0,27.0,151.0,252.00,328.0
ws_career,1742.0,18.34,29.64,-1.8,0.0,3.9,26.18,277.3
games_career,1742.0,348.35,348.58,0.0,36.0,230.5,609.00,1627.0
